In [1]:
import threading
import asyncio
import selectors
import sys

# 导入你的项目模块
from app.agents.insurance_advisor import init_insurance_agent
from app.infra.checkpointer import init_checkpointer, close_checkpointer
from langchain_core.messages import HumanMessage

# ==========================================
# 核心测试逻辑
# ==========================================
async def test_snapshot_core():
    print("⏳ 正在初始化 Checkpointer 和 Agent...")
    try:
        checkpointer = await init_checkpointer()
        agent = init_insurance_agent(checkpointer)

        config = {"configurable": {"thread_id": "snapshot_test_999"}}

        print("💬 模拟用户发送一条消息...")
        await agent.ainvoke(
            {"messages": [HumanMessage(content="达尔文12号的等待期是多久？")]},
            config=config
        )

        print("\n" + "="*30 + " 开始抓取快照 " + "="*30)
        snapshot = await agent.aget_state(config)

        print(f"📦 1. 快照对象的数据类型: {type(snapshot)}")
        print("\n📖 2. snapshot.values 里有哪些键(Key)？")
        print(list(snapshot.values.keys()))

        print("\n💬 3. 打印 messages 列表:")
        message_list = snapshot.values.get("messages", [])
        for i, msg in enumerate(message_list, 1):
            print(f"--- 第 {i} 条消息 ---")
            print(f"类型: {type(msg).__name__}")
            print(f"内容: {msg.content[:100]}...")
        print("="*75 + "\n")

    except Exception as e:
        print(f"❌ 报错了: {e}")
    finally:
        await close_checkpointer()
        print("🔒 连接池已安全关闭")

# ==========================================
# 黑科技：开一个新线程，强制使用干净的循环
# ==========================================
def run_in_new_loop():
    # 强制创建一个全新的、干净的 SelectorEventLoop
    loop = asyncio.SelectorEventLoop(selectors.SelectSelector())
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(test_snapshot_core())
    finally:
        loop.close()

print("🚀 启动后台线程运行测试...")
t = threading.Thread(target=run_in_new_loop)
t.start()
t.join() # 等待线程执行完毕
print("✅ 测试运行结束")

🚀 启动后台线程运行测试...
⏳ 正在初始化 Checkpointer 和 Agent...
2026-10-31 17:11:01 [info     ] Checkpointer初始化成功~✅️          
2026-10-31 17:11:02 [info     ] 保险顾问Agent初始化成功~✅️             
💬 模拟用户发送一条消息...
[values] {'messages': [HumanMessage(content='达尔文12号的等待期是多久？', additional_kwargs={}, response_metadata={}, id='ec16bbd8-cf88-4938-9863-1f5a152c3d3c')]}
[updates] {'model': {'messages': [AIMessage(content='我需要先查找该产品的编号，才能查询它的条款。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 1130, 'total_tokens': 1186, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 896, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 896, 'prompt_cache_miss_tokens': 234}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '8d127c26-9ffb-4e7f-83c0-fb8506878c0d', 'finish_reason': 'tool_calls',